In [ ]:
pickle 

In [1]:
import pickle

# Replace 'your_pickle_file.pkl' with the path to your pickle file
with open('averaged_coordinates.pkl', 'rb') as file:
    data = pickle.load(file)

print(data)

[(np.float64(-11.178352509111054), np.float64(18.18765385611117)), (np.float64(-11.128352509111052), np.float64(18.185431633888943)), (np.float64(-11.120574731333274), np.float64(18.18320941166672)), (np.float64(-11.088352509111054), np.float64(18.18320941166672)), (np.float64(-11.070574731333274), np.float64(18.18209830055561)), (np.float64(-11.111685842444388), np.float64(18.177653856111164)), (np.float64(-11.103908064666607), np.float64(18.176542745000052)), (np.float64(-11.073908064666606), np.float64(18.176542745000052)), (np.float64(-11.070574731333275), np.float64(18.174320522777833)), (np.float64(-11.107241397999942), np.float64(18.17209830055561)), (np.float64(-11.183908064666607), np.float64(18.169876078333388)), (np.float64(-11.15390806466661), np.float64(18.169876078333388)), (np.float64(-11.218352509111051), np.float64(18.16432052277783)), (np.float64(-11.188352509111052), np.float64(18.163209411666724)), (np.float64(-11.147241397999942), np.float64(18.163209411666724)), (

In [2]:
import numpy as np

# "coordinates" è una lista di tuple (x, y) dove x = longitudine, y = latitudine.
x_coords, y_coords = zip(*data)  # "unzip" la lista in due tuple

# Converte le tuple in array NumPy
lon_vector = np.array(x_coords)
lat_vector = np.array(y_coords)

print("Lon Vector:", lon_vector)
print("Lat Vector:", lat_vector)

Lon Vector: [-11.17835251 -11.12835251 -11.12057473 ... -10.94057473 -10.91057473
 -10.95390806]
Lat Vector: [18.18765386 18.18543163 18.18320941 ... 15.30654275 15.30654275
 15.30432052]


In [4]:
import requests
def fetch_nasa_power_data(lat, lon, start_date, end_date, parameters):
    """
    Fetches daily data from NASA POWER API for a single point.
    Parameters:
      lat, lon: Coordinates of the point.
      start_date: Start date in YYYYMMDD format.
      end_date: End date in YYYYMMDD format.
      parameters: Comma-separated list of environmental parameters.
    Returns:
      Parsed JSON response.
    """
    base_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": start_date,
        "end": end_date,
        "parameters": parameters,
        "format": "JSON"
    }
    response = requests.get(base_url, params=params)
    if response.status_code != 200:
        raise Exception(f"Error fetching data: HTTP {response.status_code}")
    return response.json()






In [13]:
import pandas as pd

# Supponiamo che tu abbia definito una funzione per ottenere il DataFrame per un singolo punto,
# ad esempio la funzione fetch_nasa_power_data() seguita dalla conversione in DataFrame.
# L'output della funzione (o una funzione wrapper) deve fornire un DataFrame in cui l'indice sono le date.
# Per esempio, supponiamo che la funzione process_point_data(coordinate, start_date, end_date, parameters)
# ritorni un DataFrame con indice (date) e le feature come colonne.

def process_point_data(coordinate, start_date, end_date, parameters):
    lat, lon = coordinate
    # Richiama la funzione che fetcha i dati (ricorda di "unpackare" coordinate se necessario).
    data = fetch_nasa_power_data(lat, lon, start_date, end_date, parameters)
    
    # I dati restituiti vengono convertiti in DataFrame (come fatto in precedenza)
    params_data = data['properties']['parameter']
    df_point = pd.DataFrame(params_data)
    df_point.index = pd.to_datetime(df_point.index, format="%Y%m%d")
    
    # Rinominazione delle colonne (aggiorna con i nomi corretti, qui 14 parametri come esempio)
    df_point.columns = [
        "Temp_media", "Temp_max", "Temp_min", "Precipitazioni", "Umidità",
        "Vento_2m", "Vento_50m", "Direzione_Vento_2m", "Pressione",
        "Radiazione_Solare", "Radiazione_LW", "Indice_Trasmissività",
        "Temp_superficie", "Mixing_Ratio"
    ]
    
    # Salva l'indice (le date) in una colonna temporanea
    df_point = df_point.reset_index().rename(columns={'index': 'data'})
    
    # Ritorna il DataFrame
    return df_point

# Definisci, ad esempio, due liste di latitudini e longitudini
lat_list = [15.0]      # esempio
lon_list = [10.0]      # esempio

# Parametri per il fetch
start_date = "20100101"
end_date = "20101231"
parameters = ",".join([
    "T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M",
    "WS2M", "WS50M", "WD2M", "PS",
    "ALLSKY_SFC_SW_DWN", "ALLSKY_SFC_LW_DWN", "ALLSKY_KT", "TS", "QV2M"
])

# Lista in cui accumuliamo i DataFrame per ogni punto
df_list = []

# Ciclo doppio per ogni coppia di coordinate
for lat, lon in zip(lat_list, lon_list):
        coordinate = (lat, lon)
        print(f"Elaborazione per coordinate: {coordinate}")
        
        # Processa i dati per il punto corrente
        df_point = process_point_data(coordinate, start_date, end_date, parameters)
        
        # Inserisce una colonna "coordinate" con (lat, lon)
        df_point.insert(0, "coordinate", [coordinate] * len(df_point))
        
        # Imposta l'indice come MultiIndex: prima livello "coordinate", secondo livello "data"
        df_point = df_point.set_index(["coordinate", "data"])
        
        # Aggiunge il DataFrame alla lista
        df_list.append(df_point)

# Concatena tutti i DataFrame in uno unico
df_final = pd.concat(df_list)

# Ora df_final ha un MultiIndex con "coordinate" (tupla) e "data" (datetime) e tutte le feature
print(df_final.head())

Elaborazione per coordinate: (15.0, 10.0)
                         Temp_media  Temp_max  Temp_min  Precipitazioni  \
coordinate   data                                                         
(15.0, 10.0) 2010-01-01       18.05     28.01     10.28             0.0   
             2010-01-02       18.28     28.46      9.90             0.0   
             2010-01-03       18.04     28.18      9.97             0.0   
             2010-01-04       17.47     27.11      9.92             0.0   
             2010-01-05       16.92     26.58      9.33             0.0   

                         Umidità  Vento_2m  Vento_50m  Direzione_Vento_2m  \
coordinate   data                                                           
(15.0, 10.0) 2010-01-01    20.37      3.75       8.24                65.7   
             2010-01-02    19.92      3.38       7.64                56.9   
             2010-01-03    17.18      2.98       6.88                50.7   
             2010-01-04    20.83      3.30     

In [ ]:
df_final